# Разметка дыхания эксперимента 3

**Статус:** активный производитель кандидатных и принятых вручную дыхательных
интервалов для записей РНЦХ. Автоматически найденные интервалы не являются
проверенной разметкой.

Порядок дыхательных режимов эксперимента 2 не используется. Каждая запись
эксперимента 3 получает собственную последовательность из первичного протокола
через внешнюю конфигурацию.


## Входы, допущения и выход

Внешняя конфигурация задаёт обезличенный `record_id`, относительный путь CSV,
роль записи, допустимые виды разметки, колонку дыхательного сигнала и
индивидуальную последовательность режимов. Постороннее несинхронное
исследование на другом приборе в список `recordings` не включается.

Алгоритм отмечает продолжительные участки с малой локальной вариабельностью,
но не присваивает им физиологические названия. Контактный артефакт, изменение
базового уровня, движение или слабое дыхание могут дать похожий участок.
Параметры эвристики сохраняются в сопроводительном файле.

Принятая разметка представляет собой упорядоченный список интервалов с
метками, точно соответствующими `mode_sequence` конкретной записи. Пустая или
неподтверждённая последовательность блокирует принятие результата.


In [ ]:
# Импорты и внешний контракт данных
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CONFIG_ENV = "KALMYKOV_EXP03_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp03_paths.example.json"
    )
CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
CSV_ROOT = (DATA_ROOT / CONFIG["csv_subdir"]).resolve()
CSV_ROOT.relative_to(DATA_ROOT)

recording_items = CONFIG["recordings"]
record_ids = [item["record_id"] for item in recording_items]
if not recording_items or len(record_ids) != len(set(record_ids)):
    raise ValueError("recordings должен содержать уникальные record_id")
BREATHING_RECORDINGS = [
    item for item in recording_items
    if "breathing" in item.get("annotation_targets", [])
]

SOURCE_COLUMNS = CONFIG["source_columns"]
CANONICAL_COLUMNS = [
    "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
    "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
]
if len(SOURCE_COLUMNS) != len(CANONICAL_COLUMNS):
    raise ValueError("source_columns должен описывать восемь столбцов CSV")

PARAMETERS = {
    key: float(value)
    for key, value in CONFIG["breathing_segmentation"].items()
}
if set(PARAMETERS) != {
    "window_s", "min_duration_s", "quiet_quantile",
    "quiet_multiplier", "quiet_floor_channel_units",
}:
    raise ValueError("breathing_segmentation содержит неверный набор параметров")
if not 0 < PARAMETERS["quiet_quantile"] < 100:
    raise ValueError("quiet_quantile задаётся в процентах от 0 до 100")
for name in (
    "window_s", "min_duration_s", "quiet_multiplier",
    "quiet_floor_channel_units",
):
    if not np.isfinite(PARAMETERS[name]) or PARAMETERS[name] <= 0:
        raise ValueError(f"{name} должен быть положительным")

OUT_DIR = DERIVED_ROOT / "exp03" / "annotations" / "breathing"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALGORITHM_VERSION = "exp03-quiet-segments-v2"


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def resolve_record_path(relative_path):
    path = (DATA_ROOT / relative_path).resolve()
    path.relative_to(CSV_ROOT)
    if not path.is_file():
        raise FileNotFoundError(f"Нет файла для записи: {relative_path}")
    return path


def read_record(path):
    frame = pd.read_csv(path)
    if list(frame.columns) != SOURCE_COLUMNS:
        raise ValueError("Схема CSV не совпадает с source_columns")
    frame.columns = CANONICAL_COLUMNS
    frame = frame.apply(pd.to_numeric, errors="raise")
    if len(frame) < 2 or not np.isfinite(frame.to_numpy(dtype=float)).all():
        raise ValueError("CSV пуст, слишком короток или содержит нечисловые значения")
    return frame


def sampling_frequency(frame):
    time = frame["time_s"].to_numpy(dtype=float)
    delta = np.diff(time)
    if len(delta) == 0 or np.any(~np.isfinite(delta)) or np.any(delta <= 0):
        raise ValueError("time_s должен быть конечным и строго возрастающим")
    median_dt = float(np.median(delta))
    jitter_fraction = float(np.median(np.abs(delta - median_dt)) / median_dt)
    return 1.0 / median_dt, jitter_fraction


In [ ]:
# Эвристический поиск спокойных участков без физиологических меток
def quiet_segments(frame, column, parameters=PARAMETERS):
    if column not in CANONICAL_COLUMNS:
        raise ValueError(f"Неизвестная колонка дыхательного сигнала: {column}")
    time = frame["time_s"].to_numpy(dtype=float)
    signal = frame[column].to_numpy(dtype=float)
    fs_hz, _ = sampling_frequency(frame)
    window = max(5, int(round(parameters["window_s"] * fs_hz)))
    rolling_std = (
        pd.Series(signal)
        .rolling(window, center=True, min_periods=max(3, window // 2))
        .std()
        .to_numpy()
    )
    finite_std = rolling_std[np.isfinite(rolling_std)]
    if len(finite_std) == 0:
        return [], np.nan
    threshold = max(
        parameters["quiet_floor_channel_units"],
        float(
            np.percentile(finite_std, parameters["quiet_quantile"])
            * parameters["quiet_multiplier"]
        ),
    )
    quiet = np.isfinite(rolling_std) & (rolling_std < threshold)
    intervals = []
    start = 0
    while start < len(quiet):
        if not quiet[start]:
            start += 1
            continue
        stop = start
        while stop < len(quiet) and quiet[stop]:
            stop += 1
        if (
            stop > start
            and time[stop - 1] - time[start] >= parameters["min_duration_s"]
        ):
            intervals.append([float(time[start]), float(time[stop - 1])])
        start = stop
    return intervals, threshold


In [ ]:
# Построение кандидатных дыхательных сопроводительных файлов
annotations = []
for spec in BREATHING_RECORDINGS:
    mode_sequence = spec.get("mode_sequence", [])
    if not mode_sequence or not all(isinstance(name, str) and name for name in mode_sequence):
        raise ValueError(
            f"Для {spec['record_id']} нужна подтверждённая mode_sequence"
        )
    source_path = resolve_record_path(spec["relative_path"])
    frame = read_record(source_path)
    fs_hz, jitter_fraction = sampling_frequency(frame)
    intervals, threshold = quiet_segments(frame, spec["respiration_column"])
    input_sha256 = sha256_file(source_path)
    output = {
        "schema_version": 2,
        "annotation_type": "breathing",
        "algorithm_version": ALGORITHM_VERSION,
        "record_id": spec["record_id"],
        "subject_id": CONFIG["subject_id"],
        "role": spec["role"],
        "input": {
            "relative_path": spec["relative_path"],
            "sha256": input_sha256,
            "sampling_frequency_hz": fs_hz,
            "sampling_frequency_source": "median_diff_time_s",
            "relative_time_step_jitter": jitter_fraction,
        },
        "respiration_column": spec["respiration_column"],
        "candidate_quiet_intervals_s": intervals,
        "protocol_mode_sequence": mode_sequence,
        "accepted_modes": None,
        "heuristic": {
            **PARAMETERS,
            "quiet_threshold_channel_units": threshold,
            "mode_labels_assigned_automatically": False,
        },
        "qc": {
            "status": "pending_manual_review",
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
            "history": [],
        },
    }
    output_path = OUT_DIR / f"{spec['record_id']}.json"
    if output_path.exists():
        existing = json.loads(output_path.read_text(encoding="utf-8"))
        if existing.get("input", {}).get("sha256") != input_sha256:
            raise RuntimeError(f"Конфликт входного SHA-256: {spec['record_id']}")
        if existing.get("algorithm_version") != ALGORITHM_VERSION:
            raise RuntimeError(
                f"Sidecar {spec['record_id']} создан другой версией алгоритма"
            )
        if (
            existing.get("protocol_mode_sequence") != mode_sequence
            or existing.get("respiration_column") != spec["respiration_column"]
        ):
            raise RuntimeError(
                f"Изменился протокольный контракт записи {spec['record_id']}"
            )
        if existing.get("qc", {}).get("status") == "accepted" and not existing.get("accepted_modes"):
            raise RuntimeError(
                f"Принятый sidecar {spec['record_id']} не содержит accepted_modes"
            )
        annotations.append(existing)
        print("Сохранён существующий sidecar:", spec["record_id"], existing["qc"]["status"])
        continue
    output_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    annotations.append(output)
    print("Создан кандидат:", spec["record_id"], len(intervals))

if len(annotations) != len(BREATHING_RECORDINGS):
    raise RuntimeError("Создан не полный набор дыхательных sidecar-файлов")
print("Дыхательных sidecar-файлов:", len(annotations))


## Ручной контроль качества и принятие разметки

Для каждой записи проверяются форма выбранного сигнала, команды первичного
протокола, состояние подключения каналов и возможные контактные или
двигательные артефакты. Ориентировочные интервалы исторического ноутбука 16
служат только навигацией.

Решение задаётся через `REVIEW_DECISION`. Статус `accepted` требует имени
проверяющего и явного списка `accepted_modes`. Повторное построение кандидатов
не перезаписывает существующий сопроводительный файл и не стирает ручной
контроль.


In [ ]:
# Ручной просмотр и явное принятие или отклонение разметки
CHECK_RECORD_ID = None
REVIEW_DECISION = None
# Пример:
# REVIEW_DECISION = {
#     "record_id": "<record_id>",
#     "status": "accepted",
#     "reviewer": "<reviewer>",
#     "accepted_modes": [
#         {"mode": "<mode-1>", "start_s": 0.0, "stop_s": 10.0},
#         {"mode": "<mode-2>", "start_s": 10.0, "stop_s": 20.0},
#     ],
#     "notes": "<основание решения>",
# }


def validate_modes(items, expected_sequence, start_s, stop_s):
    if not isinstance(items, list):
        raise ValueError("accepted_modes должен быть списком интервалов")
    labels = [item.get("mode") for item in items]
    if labels != expected_sequence:
        raise ValueError("Метки accepted_modes не совпадают с mode_sequence")
    previous_stop = start_s
    normalized = []
    for item in items:
        left = float(item["start_s"])
        right = float(item["stop_s"])
        if not np.isfinite([left, right]).all() or not start_s <= left < right <= stop_s:
            raise ValueError(f"Недопустимый интервал режима {item['mode']}")
        if left < previous_stop:
            raise ValueError("Принятые интервалы перекрываются")
        normalized.append({
            "mode": item["mode"],
            "start_s": left,
            "stop_s": right,
        })
        previous_stop = right
    return normalized


def apply_review(decision):
    record_id = decision["record_id"]
    sidecar_path = OUT_DIR / f"{record_id}.json"
    annotation = json.loads(sidecar_path.read_text(encoding="utf-8"))
    source_path = resolve_record_path(annotation["input"]["relative_path"])
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("SHA-256 исходного CSV изменился после разметки")

    status = decision["status"]
    reviewer = str(decision.get("reviewer", "")).strip()
    if status not in {"accepted", "rejected"} or not reviewer:
        raise ValueError("Нужны статус accepted/rejected и имя проверяющего")

    accepted_modes = None
    if status == "accepted":
        frame = read_record(source_path)
        time = frame["time_s"].to_numpy(dtype=float)
        accepted_modes = validate_modes(
            decision.get("accepted_modes"),
            annotation["protocol_mode_sequence"],
            float(time[0]),
            float(time[-1]),
        )

    reviewed_at = datetime.now(timezone.utc).isoformat()
    previous_qc = annotation.get("qc", {})
    history = list(previous_qc.get("history", []))
    history.append({
        "status": previous_qc.get("status"),
        "reviewer": previous_qc.get("reviewer"),
        "reviewed_at": previous_qc.get("reviewed_at"),
        "notes": previous_qc.get("notes"),
    })
    annotation["accepted_modes"] = accepted_modes
    annotation["qc"] = {
        "status": status,
        "reviewer": reviewer,
        "reviewed_at": reviewed_at,
        "notes": decision.get("notes"),
        "history": history,
    }
    sidecar_path.write_text(
        json.dumps(annotation, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return annotation


if REVIEW_DECISION is not None:
    reviewed = apply_review(REVIEW_DECISION)
    print(reviewed["record_id"], reviewed["qc"]["status"])

if CHECK_RECORD_ID is None:
    print("Задайте CHECK_RECORD_ID для просмотра конкретной записи.")
else:
    annotation = json.loads(
        (OUT_DIR / f"{CHECK_RECORD_ID}.json").read_text(encoding="utf-8")
    )
    source_path = resolve_record_path(annotation["input"]["relative_path"])
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("SHA-256 исходного CSV изменился")
    frame = read_record(source_path)
    time = frame["time_s"].to_numpy(dtype=float)
    column = annotation["respiration_column"]
    fig, axis = plt.subplots(figsize=(14, 4))
    axis.plot(time, frame[column], linewidth=0.7)
    for start, stop in annotation["candidate_quiet_intervals_s"]:
        axis.axvspan(start, stop, alpha=0.15, color="0.6")
    for item in annotation.get("accepted_modes") or []:
        axis.axvspan(
            item["start_s"], item["stop_s"], alpha=0.25, label=item["mode"]
        )
    axis.set_xlabel("Время, с")
    axis.set_ylabel(column)
    axis.set_title(
        f"{CHECK_RECORD_ID}: дыхательная разметка; "
        f"QC={annotation['qc']['status']}"
    )
    if annotation.get("accepted_modes"):
        axis.legend()
    plt.tight_layout()
    plt.show()
